In [185]:
import numpy as np
import scipy as sc

In [200]:
# Initialisation

N = 10 # Longueur du MPS
d = 2 # Nombre de paramètres physiques
chi = 5 # Bond dimension

A = [] # liste contenant le MPS 
H = [] # liste contenant le MPO

#Convention: on numérote initialement les tenseurs dans le sens anti trigo, en partant de la branche à gauche

#Ajout du premier tenseur (rang 2)
A.append(np.random.rand(d,chi))

#Ajout des tenseurs "centraux" (rang 3)
for _ in range(N-2):
    A.append(np.random.rand(chi,d,chi))

#Ajout du dernier tenseur (rang 2)
A.append(np.random.rand(chi,d))






In [201]:
#Pour les tests, on prend H nul

#Ajout du premier MPO (rang 3)
H.append(np.zeros((d,1,d)))

#Ajout des MPO "centraux" (rang 3)
for _ in range(N-2):
    H.append(np.zeros((1,d,1,d)))

#Ajout du dernier MPO (rang 2)
H.append(np.zeros((1,d,d)))



In [202]:

def MPS_orth_left(A): #Normalise le MPS à gauche en effectuant des décompositions QR successives
    M = A.copy()

    q,r = sc.linalg.qr(M[0])
    M[0] = q.copy()
    M[0].resize((d,chi))
    r_aux = r.copy()
    r_aux.resize((chi,chi))
    M[1] = np.tensordot(r_aux,M[1], axes = ([1],[0]))
   
    
    for i in range(1,N-1): #On ne s'occupe pas du dernier tenseur
        M[i] = np.reshape(M[i],(chi*d,chi))
        q,r = sc.linalg.qr(M[i], mode = "economic")
        M[i] = np.reshape(q,(chi,d,chi))
        M[i+1] = np.tensordot(r,M[i+1], axes = ([1],[0]))
        

    return M

def MPS_orth_right(A):

    M = A.copy()
    
    r,q = sc.linalg.qr(M[N-1])
    M[N-1] = q.copy()
    M[N-1].resize((chi,d))
    r_aux = r.copy()
    r_aux.resize((chi,chi))
    M[N-2] = np.tensordot(M[N-2],r_aux,1)
    

    for i in range(N-2,0,-1): #On ne s'occupe pas du premier tenseur
        
        M[i] = np.reshape(M[i],(chi,d*chi))
        r,q = sc.linalg.rq(M[i], mode = "economic")
        M[i] = np.reshape(q,(chi,d,chi))
        M[i-1] = np.tensordot(M[i-1],r,1)

    
    return M

print(MPS_orth_right(A))
        

[array([[   -83.22169677,   -405.69676867,    140.03353651,
         14207.89539654, -30747.72914325],
       [  -123.05759743,     72.39238017,    -50.85487703,
          7706.31074391, -26253.97309856]]), array([[[-1.80683665e-02,  5.02320487e-02,  2.84304256e-01,
          3.71466812e-02, -4.14066383e-03],
        [-2.35933161e-01, -8.25740806e-01,  5.38634669e-02,
          4.04715092e-01,  1.03520199e-01]],

       [[ 6.09458759e-02,  1.68048306e-01, -3.46249659e-01,
         -7.25209967e-01, -1.16180065e-01],
        [-9.81814343e-02, -3.18581793e-01, -2.32523411e-01,
         -3.46829688e-01, -1.52437902e-01]],

       [[ 5.17622174e-02,  1.57775173e-01,  7.85708793e-02,
          5.58357487e-01,  1.38709235e-01],
        [-5.30147159e-02, -2.61945149e-01, -4.38296357e-01,
         -5.90208442e-01, -1.52890244e-01]],

       [[ 7.63333000e-03,  2.40149555e-02, -1.90168728e-02,
          7.82684804e-03, -2.76093956e-01],
        [-8.63566163e-03, -1.42052781e-02, -3.61195269e-02,

In [214]:

A = MPS_orth_right(A)


def right_contraction(Hr, Mt, Mb, H):
    
    Taux = np.tensordot(Hr,Mb, axes = ([2],[2]))
    Taux = np.tensordot(Taux,H, axes = ([3,0],[3,2]))
    Taux = np.tensordot(Taux,Mt, axes = ([0,3],[0,1]))
    Taux = np.transpose(Taux,(1,2,0))  #ordre final: de bas en haut: 2,0,1 (sens horaire en partant du milieu)

    return Taux

def left_contraction(Hl,Mt,Mb,H):

    Taux = np.tensordot(Hl,Mb, axes = ([2],[0]))
    Taux = np.tensordot(Taux,H, axes = ([1,2],[0,3]))
    Taux = np.tensordot(Taux,Mt, axes = ([0,2],[2,1]))
    Taux = np.transpose(Taux,(2,1,0)) #ordre final: de haut en bas: 0, 1, 2

    return Taux

#Calcul de R[i], composante à droite de la matrice effective (Rq: on ne calcule pas R[0] ici)

R = [0]*N #liste contenant les R_i
L = [0]*N


Taux = np.tensordot(H[N-1],A[N-1], axes = ([2],[1]))
Taux = np.tensordot(Taux,A[N-1].conj().T, axes = ([1],[0]))
R[N-2] = np.transpose(Taux,(0,2,1))
print(R[N-2].shape)

Taux = np.tensordot(H[0],A[0], axes = ([2],[0]))
Taux = np.tensordot(Taux,A[0].conj().T, axes = ([0],[1]))
L[1] = np.transpose(Taux,(2,0,1))
print(L[1].shape)


for i in range(N-3,-1,-1):
    R[i] = right_contraction(R[i+1],A[i+1].conj().T,A[i+1],H[i+1])

for i in range(2,N):
    L[i] = left_contraction(L[i-1],A[i-1].conj().T,A[i-1],H[i-1])





(1, 5, 5)
(5, 1, 5)


In [216]:
print(L[1].shape)
print(R[1].shape)

def effective_Matrix(L,R,H): #Calcule la matrice à diagonaliser
        Meff = np.tensordot(L,H,axes = ([1],[0]))
        Meff = np.tensordot(Meff,R, axes = ([3],[0]))
        Meff = np.transpose(Meff, (0,2,4,1,3,5))
        Meff = np.reshape(Meff,(chi*chi*d,chi*chi*d))
        return Meff

print(effective_Matrix(L[1],R[1],H[1]))
    

(5, 1, 5)
(1, 5, 5)
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [217]:
def DMRG(N_sweeps):
    E = [] #liste des énergies

    for _ in range(N_sweeps):

        #Balayage de gauche à droite
       
        #Traitement du 1er tenseur (cas de bord: les dimensions de A[0] sont différentes)

        Meff = np.tensordot(H[0],R[0],axes = ([1],[0]))
        Meff = np.transpose(Meff, (0,2,1,3))
        Meff = np.reshape(Meff, (chi*d,chi*d))

        # Diagonalisation (Lanczos)
        val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[0]) 
        vec = np.reshape(vec,(d,chi))
        E.append(val[0])

        #Décomposition SVD sur la matrice obtenue
        U,s,V = sc.linalg.svd(vec,full_matrices = False)
        S = np.diag(s)
        V = np.matmul(S,V)

        # Mise à jour des tenseurs A[0] et A[1]
        
        A[0] = U.copy()
        A[0].resize((d,chi))
        Vaux = V.copy()
        Vaux.resize((chi,chi))
        A[1] = np.tensordot(Vaux,A[1],1)

        #Calcul de L[1] 

        Taux = np.tensordot(H[0],A[0], axes = ([2],[0]))
        Taux = np.tensordot(Taux,A[0].conj().T, axes = ([0],[1]))
        L[1] = np.transpose(Taux,(1,0,2))
        

        
        
        for i in range(1,N-1): #balayage de gauche à droite (pour les tenseurs centraux)
            Meff = effective_Matrix(L[i],R[i],H[i])

            # Diagonalisation (Lanczos)
            val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[i]) 
            vec = np.reshape(vec,(chi*d,chi))
            E.append(val[0])

            #Décomposition SVD sur la matrice obtenue
            U,s,V = sc.linalg.svd(vec,full_matrices = False)
            S = np.diag(s)
            V = np.matmul(S,V)
        
            # Mise à jour des tenseurs A[i] et A[i+1]
        
            A[i] = np.reshape(U,(chi,d,chi))
            A[i+1] = np.tensordot(V,A[i+1],1)

            #Calcul de L[i] 
            L[i+1] = left_contraction(L[i],A[i].conj().T,A[i],H[i])

        #Traitement du dernier tenseur


        #Balayage de droite à gauche
        
        #Traitement du dernier tenseur
        
        Meff = np.tensordot(L[N-1],H[N-1],axes = ([1],[0]))
        Meff = np.transpose(Meff, (0,2,1,3))
        Meff = np.reshape(Meff, (chi*d,chi*d))

        # Diagonalisation (Lanczos)
        val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[0]) 
        vec = np.reshape(vec,(d,chi))
        E.append(val[0])

        # Décomposition SVD sur la matrice obtenue
        U,s,V = sc.linalg.svd(vec,full_matrices = False)
        S = np.diag(s)
        V = np.matmul(S,V)

        # Mise à jour des tenseurs A[N-1] et A[N-2]
        
        A[N-1] = V.copy()
        A[N-1].resize((chi,d))
        Uaux = U.copy()
        Uaux.resize((chi,chi))
        A[N-2] = np.tensordot(A[N-2],Uaux,1)
    

        #Calcul de R[N-2] 
        Taux = np.tensordot(H[N-1],A[N-1], axes = ([2],[1]))
        Taux = np.tensordot(Taux,A[N-1].conj().T, axes = ([1],[0]))
        R[N-2] = np.transpose(Taux,(0,2,1))

        

        for i in range(N-2,0,-1): #balayage de droite à gauche (pour les MPS centraux)
    
            Meff = effective_Matrix(L[i],R[i],H[i])

            # Diagonalisation (Lanczos)
            val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[i]) 
            vec = np.reshape(vec,(chi,d*chi))
            E.append(val[0])

            #Décomposition SVD sur la matrice obtenue
            U,s,V = sc.linalg.svd(vec,full_matrices = False)
            S = np.diag(s)
            U = np.matmul(U,S)
        
            # Mise à jour des tenseurs A[i] et A[i+1]
        
            A[i] = np.reshape(V,(chi,d,chi))
            A[i-1] = np.tensordot(A[i-1],U,1)

            #Calcul de R[i-1] 
            R[i-1] = right_contraction(R[i],A[i].conj().T,A[i],H[i])
        
    return E

print(DMRG(10))

        
        

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
